In [ ]:
'''
Train a Transformer encoder model to generate MIDI in two parts:
    1. pre-train
    (2. SFT)
    (3. RL)
'''
from pathlib import Path
from random import shuffle
import pickle
from miditok import REMI, TokenizerConfig
from preprocess import create_bar_dataset
from config import ModelParams, HyperParams

PREPROCESS = False # set to True if you haven't run preprocess yet
TRAIN = False # set to False if you want to skip training

TOKENIZER_PARAMS = {
    "special_tokens": ["PAD_None", "MASK_None"],
    "pitch_range": (22, 107),
    "num_velocities": 24,
    "beat_res": {(0, 8): 4}, # 16 Position tokens within a bar
    "use_pitchdrum_tokens": False,
    "use_chords": False,
    "chord_tokens_with_root_note": False,
}
config = TokenizerConfig(**TOKENIZER_PARAMS)
tokenizer = REMI(config)

if PREPROCESS:
    midi_paths = [f for d in Path(HyperParams.DATA_PATH).iterdir() if d.is_dir() for f in d.glob("*.mid")]
    shuffle(midi_paths)
    midi_paths = [p.resolve() for p in midi_paths if p.is_file()]
    total_num_files = len(midi_paths)

    tokenizer.save(Path("models", HyperParams.name + "_tokenizer.json"))

    num_files_valid = round(total_num_files * 0.05)
    num_files_test = round(total_num_files * 0.05)
    midi_paths_val = midi_paths[:num_files_valid]
    midi_paths_test = midi_paths[num_files_valid:num_files_valid + num_files_test]
    midi_paths_train = midi_paths[num_files_valid + num_files_test:]
    
    create_bar_dataset(midi_paths_train, tokenizer, Path("SFT_dataset", "train"))
    create_bar_dataset(midi_paths_val, tokenizer, Path("SFT_dataset", "val"))
    create_bar_dataset(midi_paths_test, tokenizer, Path("SFT_dataset", "test"))

    print(f"Dataset created in {Path("midi_dataset", "train").absolute()}")
    
else:
    tokenizer._load_from_json(Path("models", HyperParams.name + "_tokenizer.json"))

# Load a sample to check its content
if len(list(Path("SFT_dataset", "train").glob("*.pkl"))) > 0:
    sample_file = list(Path("SFT_dataset", "train").glob("*.pkl"))[0]
    with open(sample_file, "rb") as f:
        data = pickle.load(f)
        print("--- Sample Verification ---")
        print(f"Loaded sample: {sample_file.name}")
        print(f"Token IDs (first 20): {data['input_ids'][:20]}...")
        print("-------------------------")

print(tokenizer.vocab)
print(tokenizer.vocab_size)

In [ ]:
from preprocess import MIDIDataset, DataCollator
from torch.utils.data import DataLoader

midi_paths_train = list(Path("SFT_dataset", "train").rglob("*.pkl"))
midi_paths_val = list(Path("SFT_dataset", "val").rglob("*.pkl"))

train_dataset = MIDIDataset(midi_paths_train, tokenizer)
val_dataset = MIDIDataset(midi_paths_val, tokenizer)

collator = DataCollator(pad_token_id=tokenizer.vocab['PAD_None'])

train_dataloader = DataLoader(dataset=train_dataset, batch_size=HyperParams.batch_size, collate_fn=collator, shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=HyperParams.batch_size, collate_fn=collator, shuffle=True)

import torch
for batch in train_dataloader:
    '''
    length = batch['input_ids'].shape[1]
    if max_len < length:
        max_len = length
    '''
    print(batch['input_ids'].shape)
    print(batch['input_length'])
    print(batch['length'])
    print(batch['length'].max().item())

    b = batch['input_ids'].shape[1]
    t = torch.rand((b,), device=batch['input_ids'].device)
    print(t.shape)
    t = torch.rand(b, device=batch['input_ids'].device)
    print(t.shape)

    prompt = batch['input_ids'][0][:batch['input_length'][0]]
    answer = batch['input_ids'][0][batch['input_length'][0]:]
    test = tokenizer.decode([prompt, answer])
    print(test)
    test.dump_midi(Path("midi_start.mid"))
    break

In [ ]:
from help import Trainer
import torch
from tqdm.notebook import tqdm

trainer = Trainer(ModelParams, HyperParams, tokenizer)

# Setup
trainer.phase = 'SFT' # 'pre-training' or 'SFT'
model_dir = 'models/checkpoints/' + trainer.phase + '_'

print(sum(p.numel() for p in trainer.mask_predictor.parameters()), "model parameters\n")
print("Model architecture:")
print(trainer.mask_predictor)

trainer.load("models/checkpoints/SFT_72000_checkpoint.pth", True)

In [ ]:
if TRAIN:
    from torch.utils.tensorboard import SummaryWriter
    from itertools import cycle
    
    writer = SummaryWriter()
    global_step = 0
    # Optional: wrap dataloader with cycle to avoid worrying about epoch ends
    train_iterator = cycle(train_dataloader)

    pbar = tqdm(total=HyperParams.num_training_steps, desc="Training", position=0)

    best_val = float('inf')
    train_loss = 0

    trainer.mask_predictor.train()
    while global_step < HyperParams.num_training_steps:
        train_batch = next(train_iterator)

        train_loss0 = trainer.train_step(train_batch)
        train_loss += train_loss0

        # Logging
        global_step += 1
        if global_step % HyperParams.logging_interval == 0:
            trainer.mask_predictor.eval()

            val_loss = 0
            for val_batch in tqdm(val_dataloader, desc="Validating", leave=False):
                val_loss += trainer.val_step(val_batch)

            if val_loss < best_val:
                trainer.save('models/best_val', global_step, val_loss / len(val_dataloader))
                best_val = val_loss

            writer.add_scalar('Loss/val', val_loss / len(val_dataloader), global_step)
            writer.add_scalar('Loss/train', train_loss / HyperParams.logging_interval, global_step)
            writer.add_scalar('LR', trainer.lr_scheduler.get_last_lr()[0], global_step)
            train_loss = 0

            trainer.save(model_dir + str(global_step), global_step, val_loss)

            trainer.mask_predictor.train()
        pbar.update(1)

    pbar.close()
    writer.close()